## test

In [20]:
import h3 as h3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
# from keplergl import KeplerGl
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [21]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [9]:
# import numpy as np
# import pandas as pd
# from pyhive import presto
# from datetime import timedelta,datetime
# import json
# import requests
# import math
# import os


# pd.set_option('display.max_columns', None)
# import warnings
# warnings.filterwarnings("ignore")
conn = presto.connect(
        host="presto-gateway.serving.data.plectrum.dev",
        port=80,
        username="manoj.ravirajan@rapido.bike"
    )
presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)

In [19]:
fe_query = '''
    select
        cluster,
        hex_id,
        executiondate
    from 
            datasets.city_cluster_hex
        where
            city = 'Bangalore'
            and 
                resolution = 8
'''

In [7]:
test_df = pd.read_sql(fe_query, connection)

In [8]:
test_df

,cluster,hex_id,executiondate
0,Nelamangala,8860144847fffff,2023-09-03
1,Bidadi,8860144029fffff,2023-09-03
2,Bangalore Airport,886016908bfffff,2023-09-03
3,Bannerghatta,886014535dfffff,2023-09-03
4,Bannerghatta,88618926d5fffff,2023-09-03
...,...,...,...
3225,Attibele,886014cb65fffff,2023-09-03
3226,Attibele,88618936a3fffff,2023-09-03
3227,Attibele,88618936b1fffff,2023-09-03
3228,Attibele,88618936b7fffff,2023-09-03


In [1]:
print('523452')

523452


In [23]:
sample_query2 = """

select 
    DISTINCT
    data__cityname city_name, data__featuretype type, data__name cluster_name
from 
    hive.canonical.iceberg_domain_geo_hex_layer_immutable 
where 
    data__featuretype in ('cluster', 'city')
order by 1, 2, 3

"""

In [24]:
view = """

-- select * from reports_internal.rtu_customer_rf_recos_dec10_view
select * from  reports_internal.rtu_customer_rf_recos_dec10_v3
-- rtu_customer_rf_recos_dec10_query2_view


"""

In [25]:

df2 = pd.read_sql(sample_query2, connection)
print(df2.shape)
df2.head()


(11331, 3)


,city_name,type,cluster_name
0,Abohar,city,Abohar
1,Abohar,cluster,Alamgarh
2,Abohar,cluster,Army Cantt
3,Abohar,cluster,Azimgarh
4,Abohar,cluster,Daulat Pura


In [8]:
df2.groupby(['city_name']).agg(cluster_count=('cluster_name', 'nunique')).sort_values(by='cluster_count', ascending=False).reset_index()

,city_name,cluster_count
0,Delhi,544
1,Kolkata,488
2,Hyderabad,313
3,Mumbai,275
4,Siliguri,267
...,...,...
296,Itarsi,3
297,Shravasti,2
298,Bhuj,2
299,Nagda,2


In [26]:

# df2.to_parquet('RF_data.parquet',index=False)
df2.to_csv('city_cluster_mapping.csv',index=False)

In [10]:
def run_bookingtime_query(yyyymmdd):
    
    qq = f""" 
    with ao as (
    
        select 
            yyyymmdd,
            -- platform,
            user_id,
            CAST(CAST(ct_session_id AS DECIMAL) AS VARCHAR) || ' - ' || phone AS ao_session_id,
            min(epoch) AS ao_epoch
        from 
            clevertap.clevertap_customer_order_activity
        where 
            yyyymmdd = '{yyyymmdd}'
            and current_city = 'Bangalore'
            and serviceable = 'true'
            and order_activity_source = 'appOpen'
        group by 1,2,3
    
        ),
        
        fe as (
        
        select 
            yyyymmdd,
            user_id,
            CAST(CAST(ct_session_id AS DECIMAL) AS VARCHAR) || ' - ' || phone AS fe_session_id,
            min(epoch) AS fe_epoch
        from 
            canonical.clevertap_customer_fare_estimate
        where 
            yyyymmdd = '{yyyymmdd}'
            and current_city = 'Bangalore'
        group by 1,2,3
        ),
        
        rr as (
        
        select 
            yyyymmdd,
            user_id,
            CAST(CAST(ct_session_id AS DECIMAL) AS VARCHAR) || ' - ' || phone AS rr_session_id,
            min(epoch) AS rr_epoch
        from 
            canonical.clevertap_customer_request_rapido
        where 
            yyyymmdd = '{yyyymmdd}'
            and current_city = 'Bangalore'
        group by 1,2,3
        ),
        
        ao_fe_rr as (
        
        select
            ao.*,
            fe.fe_session_id,
            fe.fe_epoch,
            rr.rr_session_id,
            rr.rr_epoch,
            ((fe.fe_epoch - ao.ao_epoch)/1000) ao2fe,
            ((rr.rr_epoch - fe.fe_epoch)/1000) fe2rr,
            ((rr.rr_epoch - ao.ao_epoch)/1000) ao2rr
        from 
            ao
        left join 
            fe
            on ao.yyyymmdd = fe.yyyymmdd
            and ao.user_id = fe.user_id
            and ao.ao_session_id = fe.fe_session_id
        left join 
            rr
            on fe.yyyymmdd = rr.yyyymmdd
            and fe.user_id = rr.user_id
            and fe.fe_session_id = rr.rr_session_id
        )
        
        select
            *
        from
            ao_fe_rr
    """
    return pd.read_sql(qq, conn1)

In [18]:
from datetime import datetime, timedelta
from tqdm import tqdm

# Define the start date in 'YYYYMMDD' format
start_date_str = '20250310'
start_date = datetime.strptime(start_date_str, '%Y%m%d')

# Generate all dates between start_date and current_date
dates = [(start_date + timedelta(days=i)).strftime('%Y%m%d') for i in range(10)]
dates

['20250310',
 '20250311',
 '20250312',
 '20250313',
 '20250314',
 '20250315',
 '20250316',
 '20250317',
 '20250318',
 '20250319']

In [19]:
datalist = []
for yyyymmdd in tqdm(dates):
    datalist.append(run_bookingtime_query(yyyymmdd))

 20%|██        | 2/10 [02:06<08:27, 63.42s/it]


KeyboardInterrupt: 

In [ ]:
datalist

In [13]:
df = pd.concat(datalist, ignore_index=True)

In [15]:
df.to_parquet('booking_time_mar20_apr02.parquet')

In [22]:
df_delhi = pd.read_parquet('/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/df_20250303.parquet')

In [24]:
df_delhi.shape

(11141880, 58)

In [25]:
df_delhi1 = pd.read_parquet('/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/segment/test/df_20250303.parquet')

In [26]:
df_delhi1.shape

(9106568, 4)

In [27]:
df_delhi.head(5)

,customer_id,taxi_ltr_segment,rapido_firstride_age,customerid,max_active_days,total_active_days,isAct_w01,isAct_w02,isAct_w03,isAct_w04,...,hh_quarterly_segment,hh_inactive_segment,daily_segment,weekly_segment,monthly_segment,quarterly_segment,inactive_segment,new_segment,segment,updated_segment
0,63145a0640c18148020ff5e5,HH,911,NA,0.0,0.0,0.0,0.0,0.0,0.0,...,None,3-HH-INACTIVE,None,None,None,None,None,None,3-HH-INACTIVE,3-HH-INACTIVE
1,65953febb6c2d0503fbb8c1c,HH,425,65953febb6c2d0503fbb8c1c,91.0,1.0,0.0,0.0,0.0,0.0,...,2-HH-QUARTERLY,3-HH-INACTIVE,None,None,None,None,None,None,2-HH-QUARTERLY,2-HH-QUARTERLY
2,670da09b0401f7fbd582e8ba,HH,139,NA,0.0,0.0,0.0,0.0,0.0,0.0,...,None,3-HH-INACTIVE,None,None,None,None,None,None,3-HH-INACTIVE,3-HH-INACTIVE
3,67205781aa7eb8928ebbe454,HH,125,NA,0.0,0.0,0.0,0.0,0.0,0.0,...,None,3-HH-INACTIVE,None,None,None,None,None,None,3-HH-INACTIVE,3-HH-INACTIVE
4,62fb9bc8c496e55cdb09a787,HH,247,62fb9bc8c496e55cdb09a787,91.0,0.0,0.0,0.0,0.0,0.0,...,None,3-HH-INACTIVE,None,None,None,None,None,None,3-HH-INACTIVE,3-HH-INACTIVE
